# Notebook 3: Huấn luyện Conditioned Music Decoder
**Freeze:** GestureEmotionEncoder + MusicPrior
**Train:** Chỉ Cross-Attention weights trong ConditionedDecoder
**Output:** conditioned_decoder.pt + ảnh đánh giá

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import os, math
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family':'serif','font.size':11,'axes.titlesize':12,
    'axes.spines.top':False,'axes.spines.right':False,'axes.grid':True,'grid.alpha':0.3,'grid.linestyle':'--'})
SAVE='/kaggle/working/'
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')

for p in ['/kaggle/input/datasets/quangleuq/v3-data/','/kaggle/input/gesturhythm-v2/']:
    if os.path.exists(p+'sequences.npy'): DATA_DIR=p; break
X=np.load(DATA_DIR+'sequences.npy'); y=np.load(DATA_DIR+'emotions.npy'); m=np.load(DATA_DIR+'mode_ids.npy')
print(f'X:{X.shape} | y:{y.shape}')
MODE={0:'HAPPY HIGH',1:'HAPPY LOW',2:'SAD HIGH',3:'SAD LOW',4:'NEUTRAL',5:'NONE'}
COLORS={0:'#27ae60',1:'#2980b9',2:'#c0392b',3:'#8e44ad',4:'#e67e22',5:'#7f8c8d'}


## 1. Load Pretrained Models (Frozen)

In [ ]:
# ── Model definitions ──────────────────────────────────────────────
class GestureEmotionEncoder(nn.Module):
    def __init__(self,input_dim=225,d_model=128,nhead=4,num_layers=3):
        super().__init__()
        self.embed=nn.Linear(input_dim,d_model); self.pos_enc=nn.Embedding(512,d_model)
        enc=nn.TransformerEncoderLayer(d_model=d_model,nhead=nhead,dim_feedforward=256,dropout=0.0,batch_first=True)
        self.transformer=nn.TransformerEncoder(enc,num_layers=num_layers)
        self.head=nn.Sequential(nn.Linear(d_model,64),nn.ReLU(),nn.Linear(64,2),nn.Tanh())
    def forward(self,x):
        B,T,_=x.shape
        mask=torch.triu(torch.ones(T,T,device=x.device),diagonal=1).bool()
        pos=torch.arange(T,device=x.device).unsqueeze(0)
        return self.head(self.transformer(self.embed(x)+self.pos_enc(pos),mask=mask)[:,-1,:])

class MusicPrior(nn.Module):
    def __init__(self,vocab=130,d_model=128,nhead=4,num_layers=4,max_len=512):
        super().__init__()
        self.embed=nn.Embedding(vocab,d_model,padding_idx=129); self.pos_enc=nn.Embedding(max_len,d_model)
        enc=nn.TransformerEncoderLayer(d_model=d_model,nhead=nhead,dim_feedforward=256,dropout=0.0,batch_first=True)
        self.transformer=nn.TransformerEncoder(enc,num_layers=num_layers); self.head=nn.Linear(d_model,vocab)
    def get_hidden(self,tokens):
        B,T=tokens.shape
        mask=torch.triu(torch.ones(T,T,device=tokens.device),diagonal=1).bool()
        pos=torch.arange(T,device=tokens.device).unsqueeze(0)
        return self.transformer(self.embed(tokens)+self.pos_enc(pos),mask=mask)

# ── Load frozen models ─────────────────────────────────────────────
MODEL_DIR='/kaggle/input/gesturhythm-v2-models/'
enc_ckpt=torch.load(MODEL_DIR+'gesture_emotion_encoder.pt',map_location='cpu',weights_only=False)
prior_ckpt=torch.load(MODEL_DIR+'music_prior.pt',map_location='cpu',weights_only=False)

encoder=GestureEmotionEncoder(**enc_ckpt['config']).to(device)
encoder.load_state_dict(enc_ckpt['model_state']); encoder.eval()
for p in encoder.parameters(): p.requires_grad=False

prior=MusicPrior(**prior_ckpt['config']).to(device)
prior.load_state_dict(prior_ckpt['model_state']); prior.eval()
for p in prior.parameters(): p.requires_grad=False

print('GestureEmotionEncoder: FROZEN')
print('MusicPrior: FROZEN')
enc_metrics=enc_ckpt.get('metrics',{})
prior_metrics=prior_ckpt.get('metrics',{})
print(f'Encoder val_loss={enc_metrics.get("val_loss","N/A")}')
print(f'Prior perplexity={prior_metrics.get("perplexity","N/A")}')


## 2. Conditioned Decoder (Module duy nhất được train)

In [ ]:
class ConditionedDecoder(nn.Module):
    """
    Cross-Attention: emotion_vec -> Key/Value, prior_hidden -> Query.
    Chi train phan nay; encoder va prior bi dong bang (frozen).
    """
    def __init__(self,emotion_dim=2,d_model=128,nhead=4,num_layers=2,vocab=130):
        super().__init__()
        self.emotion_proj=nn.Linear(emotion_dim,d_model)
        dec=nn.TransformerDecoderLayer(d_model=d_model,nhead=nhead,dim_feedforward=256,dropout=0.1,batch_first=True)
        self.transformer=nn.TransformerDecoder(dec,num_layers=num_layers)
        self.head=nn.Linear(d_model,vocab)
    def forward(self,emotion_vec,prior_hidden):
        memory=self.emotion_proj(emotion_vec).unsqueeze(1)  # (B,1,d_model) = Key/Value
        out=self.transformer(prior_hidden,memory)           # prior_hidden = Query
        return self.head(out)                               # (B,T,vocab)

decoder=ConditionedDecoder().to(device)
trainable=sum(p.numel() for p in decoder.parameters())
total_enc=sum(p.numel() for p in encoder.parameters())
total_prior=sum(p.numel() for p in prior.parameters())
print(f'ConditionedDecoder (trainable): {trainable:,}')
print(f'Encoder (frozen):              {total_enc:,}')
print(f'MusicPrior (frozen):           {total_prior:,}')
print(f'Ty le trainable: {trainable/(trainable+total_enc+total_prior)*100:.1f}%')


## 3. Dataset và Data Loader

In [ ]:
SCALES={'major':[60,62,64,65,67,69,71,72,74,76],'minor':[60,62,63,65,67,68,70,72,74,75],
        'pentatonic':[60,62,64,67,69,72,74,76]}

def emotion_to_token_seq(val,aro,seq_len=32):
    scale=SCALES['major'] if val>0.2 else (SCALES['pentatonic'] if abs(val)<0.2 else SCALES['minor'])
    center=67 if val>0.2 and aro>0 else (60 if val<-0.2 else 64)
    tokens=[]; prev=center
    for _ in range(seq_len):
        cands=[n for n in scale if abs(n-prev)<=5] or scale
        nxt=cands[np.random.randint(len(cands))]
        if aro<-0.3 and np.random.random()<0.2: tokens.append(128)
        tokens.append(nxt); prev=nxt
    return tokens[:seq_len]

class CondDataset(Dataset):
    SEQ=32
    def __init__(self,X,y):
        self.X=torch.tensor(X,dtype=torch.float32)
        self.y=torch.tensor(y,dtype=torch.float32)
        self.tgts=[emotion_to_token_seq(float(y[i,0]),float(y[i,1]),self.SEQ+1) for i in range(len(y))]
    def __len__(self): return len(self.X)
    def __getitem__(self,i):
        t=self.tgts[i]
        return (self.X[i],self.y[i],
                torch.tensor(t[:-1],dtype=torch.long),
                torch.tensor(t[1:],dtype=torch.long))

ds=CondDataset(X,y); n=int(0.8*len(ds))
train_ds,val_ds=random_split(ds,[n,len(ds)-n],generator=torch.Generator().manual_seed(42))
train_loader=DataLoader(train_ds,batch_size=32,shuffle=True)
val_loader=DataLoader(val_ds,batch_size=32)
print(f'Train:{n} | Val:{len(ds)-n}')


## 4. Huấn luyện

In [ ]:
optimizer=torch.optim.Adam(decoder.parameters(),lr=1e-3)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=50)
criterion=nn.CrossEntropyLoss(ignore_index=129)
EPOCHS=50; train_losses,val_losses,perplexities=[],[],[]

for epoch in range(EPOCHS):
    decoder.train(); tl=0
    for gs,ev,ti,to in train_loader:
        gs,ev,ti,to=gs.to(device),ev.to(device),ti.to(device),to.to(device)
        with torch.no_grad():
            pred_emo=encoder(gs); prior_h=prior.get_hidden(ti)
        optimizer.zero_grad()
        logits=decoder(pred_emo,prior_h)
        loss=criterion(logits.reshape(-1,130),to.reshape(-1))
        loss.backward(); torch.nn.utils.clip_grad_norm_(decoder.parameters(),1.0)
        optimizer.step(); tl+=loss.item()
    decoder.eval(); vl=0
    with torch.no_grad():
        for gs,ev,ti,to in val_loader:
            gs,ev,ti,to=gs.to(device),ev.to(device),ti.to(device),to.to(device)
            pred_emo=encoder(gs); prior_h=prior.get_hidden(ti)
            vl+=criterion(decoder(pred_emo,prior_h).reshape(-1,130),to.reshape(-1)).item()
    scheduler.step()
    tl_a=tl/len(train_loader); vl_a=vl/len(val_loader)
    train_losses.append(tl_a); val_losses.append(vl_a); perplexities.append(math.exp(vl_a))
    if (epoch+1)%10==0:
        print(f'Epoch {epoch+1:2d}/{EPOCHS} | train={tl_a:.4f} | val={vl_a:.4f} | ppl={math.exp(vl_a):.2f}')
print('Done!')


## 5. Đánh giá định lượng và biểu đồ học thuật

In [ ]:
fig,axes=plt.subplots(2,3,figsize=(15,9))
fig.suptitle('Hình T3.1: Kết quả huấn luyện Conditioned Decoder',fontsize=13,fontweight='bold')

# (a) Loss curves
axes[0,0].plot(train_losses,color='steelblue',lw=2,label='Train')
axes[0,0].plot(val_losses,color='coral',lw=2,label='Val')
axes[0,0].set_title('(a) Loss (CrossEntropy)'); axes[0,0].set_xlabel('Epoch'); axes[0,0].set_ylabel('Loss')
axes[0,0].legend()

# (b) Perplexity
axes[0,1].plot(perplexities,color='mediumpurple',lw=2)
axes[0,1].set_title('(b) Perplexity'); axes[0,1].set_xlabel('Epoch')
axes[0,1].annotate(f'Final:{perplexities[-1]:.2f}',xy=(len(perplexities)-1,perplexities[-1]),
    xytext=(-30,-20),textcoords='offset points',arrowprops=dict(arrowstyle='->',color='red'),color='red',fontsize=9)

# (c) & (d) Sinh melody 2 cam xuc doi lap
decoder.eval()
EMOTIONS={'Happy (+0.8,+0.8)':torch.tensor([[0.8,0.8]]),'Sad (-0.8,-0.8)':torch.tensor([[-0.8,-0.8]]),
          'Tense (-0.8,+0.8)':torch.tensor([[-0.8,0.8]]),'Calm (+0.8,-0.8)':torch.tensor([[0.8,-0.8]])}

def gen_melody(emo_vec,n=24):
    tokens=torch.tensor([[60,64,67]],dtype=torch.long).to(device)
    emo=emo_vec.to(device)
    notes=[]
    for _ in range(n):
        h=prior.get_hidden(tokens)
        logits=decoder(emo,h)[0,-1,:].clone()
        logits[128:]=float('-inf')
        nxt=torch.multinomial(F.softmax(logits/0.8,dim=0),1)
        tokens=torch.cat([tokens,nxt.unsqueeze(0)],dim=1)
        if nxt.item()<128: notes.append(nxt.item())
    return notes

colors_emo=['#27ae60','#c0392b','#8e44ad','#2980b9']
for ai,(label,emo) in zip([(0,2),(1,3)],list(EMOTIONS.items())[:2]):
    notes=gen_melody(emo)
    c=colors_emo[list(EMOTIONS.keys()).index(label)]
    axes[0,ai+1].step(range(len(notes)),notes,where='post',color=c,lw=2)
    axes[0,ai+1].fill_between(range(len(notes)),notes,step='post',alpha=0.2,color=c)
    axes[0,ai+1].set_title(f'Melody sinh ra\n{label}')
    axes[0,ai+1].set_xlabel('Vi tri not'); axes[0,ai+1].set_ylabel('MIDI pitch')
    axes[0,ai+1].set_ylim(48,88)

# (e) So sanh pitch trung binh 4 cam xuc
all_notes={}
for label,emo in EMOTIONS.items():
    n=[gen_melody(emo) for _ in range(5)]
    all_notes[label]=np.concatenate(n)

labels_e=list(all_notes.keys())
means=[np.mean(v) for v in all_notes.values()]
stds=[np.std(v) for v in all_notes.values()]
bars=axes[1,0].bar(range(len(labels_e)),means,color=colors_emo,edgecolor='black',lw=0.8)
axes[1,0].errorbar(range(len(labels_e)),means,yerr=stds,fmt='none',color='black',capsize=5,lw=1.5)
axes[1,0].set_xticks(range(len(labels_e))); axes[1,0].set_xticklabels([l.split('(')[0] for l in labels_e],rotation=20,ha='right',fontsize=9)
axes[1,0].set_title('(e) Pitch trung binh theo cảm xúc'); axes[1,0].set_ylabel('MIDI pitch TB')

# (f) Histogram pitch 2 cam xuc doi lap
notes_happy=all_notes['Happy (+0.8,+0.8)']
notes_sad=all_notes['Sad (-0.8,-0.8)']
axes[1,1].hist(notes_happy,bins=20,alpha=0.6,color='#27ae60',label='Happy',density=True,edgecolor='none')
axes[1,1].hist(notes_sad,bins=20,alpha=0.6,color='#c0392b',label='Sad',density=True,edgecolor='none')
axes[1,1].set_title('(f) Phân bố pitch: Happy vs Sad')
axes[1,1].set_xlabel('MIDI pitch'); axes[1,1].set_ylabel('Mật độ'); axes[1,1].legend()

# (g) Piano roll happy vs sad
ax=axes[1,2]
notes_h=gen_melody(list(EMOTIONS.values())[0],n=16)
notes_s=gen_melody(list(EMOTIONS.values())[1],n=16)
n_show=min(len(notes_h),len(notes_s),16)
ax.step(range(n_show),notes_h[:n_show],where='post',color='#27ae60',lw=2,label='Happy')
ax.step(range(n_show),notes_s[:n_show],where='post',color='#c0392b',lw=2,label='Sad',linestyle='--')
ax.set_title('(g) So sánh melody Happy vs Sad')
ax.set_xlabel('Vị trí nốt'); ax.set_ylabel('MIDI pitch'); ax.legend(); ax.set_ylim(48,88)

plt.tight_layout()
plt.savefig(SAVE+'dec_fig1_training_results.png',dpi=150,bbox_inches='tight'); plt.show()
print('Saved dec_fig1_training_results.png')
import scipy.stats as ss
stat,pval=ss.ttest_ind(notes_happy,notes_sad)
print(f'T-test Happy vs Sad: t={stat:.3f}, p={pval:.4f} ({'co y nghia' if pval<0.05 else 'khong y nghia'} thong ke)')


## 6. Lưu model

In [ ]:
torch.save({
    'model_state':decoder.state_dict(),
    'config':{'emotion_dim':2,'d_model':128,'nhead':4,'num_layers':2,'vocab_size':130},
    'metrics':{'val_loss':val_losses[-1],'perplexity':perplexities[-1]},
    'train_losses':train_losses,'val_losses':val_losses
},'conditioned_decoder.pt')
print('Saved: conditioned_decoder.pt')
print(f'Val Loss: {val_losses[-1]:.4f} | Perplexity: {perplexities[-1]:.2f}')
